# MCS603 — Programming II (Python)
## Week 12–13: Database Access — SQLite & MySQL

---
### Learning Objectives
- Understand relational database concepts: tables, rows, columns, keys
- Connect to SQLite using Python's built-in `sqlite3` module
- Perform full CRUD operations (Create, Read, Update, Delete)
- Use parameterised queries to prevent SQL injection
- Understand how to connect to MySQL with `mysql-connector-python`
- Build a database-driven student management system

> **Setup:** SQLite is built into Python. For MySQL: `pip install mysql-connector-python`

### Outline
1. Database Concepts  
2. SQLite with `sqlite3`  
3. Creating Tables  
4. CRUD Operations  
5. Parameterised Queries & SQL Injection  
6. Transactions  
7. Querying with `fetchone` / `fetchall`  
8. MySQL Connector  
9. Full Student Management System  
10. Lab Exercises  
11. Assignment  

---
## 1. Database Concepts

| Term | Meaning |
|---|---|
| **Table** | Grid of rows and columns (like a spreadsheet) |
| **Row / Record** | One entry in the table |
| **Column / Field** | One attribute shared by all rows |
| **Primary Key (PK)** | Unique identifier for a row |
| **Foreign Key (FK)** | Reference to a PK in another table |
| **CRUD** | Create, Read, Update, Delete |
| **SQL** | Structured Query Language — universal database language |

### CRUD ↔ SQL
| Operation | SQL Statement |
|---|---|
| Create | `INSERT INTO table VALUES (...)` |
| Read | `SELECT * FROM table WHERE ...` |
| Update | `UPDATE table SET col=val WHERE ...` |
| Delete | `DELETE FROM table WHERE ...` |

---
## 2. SQLite with `sqlite3`

SQLite stores the entire database in a single `.db` file — no server needed.  
Python ships with `sqlite3` in the standard library.

```python
import sqlite3

conn   = sqlite3.connect("database.db")  # creates file if missing
cursor = conn.cursor()                    # execute SQL via cursor
# ... do work ...
conn.commit()    # save changes
conn.close()     # free connection
```

Use `sqlite3.connect(":memory:")` for a temporary in-memory database (great for testing).

In [ ]:
import sqlite3

conn   = sqlite3.connect("school.db")
cursor = conn.cursor()

# Return rows as dicts instead of tuples
conn.row_factory = sqlite3.Row

print("Connected to school.db")
print("SQLite version:", sqlite3.sqlite_version)

---
## 3. Creating Tables

In [ ]:
# DROP and recreate for a clean slate (demo purposes)
cursor.executescript("""
    DROP TABLE IF EXISTS enrollments;
    DROP TABLE IF EXISTS courses;
    DROP TABLE IF EXISTS students;

    CREATE TABLE students (
        id         INTEGER PRIMARY KEY AUTOINCREMENT,
        name       TEXT    NOT NULL,
        email      TEXT    UNIQUE NOT NULL,
        dob        DATE,
        created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
    );

    CREATE TABLE courses (
        code     TEXT    PRIMARY KEY,
        title    TEXT    NOT NULL,
        credits  INTEGER NOT NULL DEFAULT 4,
        capacity INTEGER NOT NULL DEFAULT 30
    );

    CREATE TABLE enrollments (
        id         INTEGER PRIMARY KEY AUTOINCREMENT,
        student_id INTEGER NOT NULL REFERENCES students(id),
        course_code TEXT   NOT NULL REFERENCES courses(code),
        grade      INTEGER CHECK(grade BETWEEN 0 AND 100),
        UNIQUE(student_id, course_code)
    );
""")
conn.commit()
print("Tables created.")

---
## 4. CRUD Operations

### CREATE — INSERT

In [ ]:
# Insert courses
courses = [
    ("MCS601", "Programming I",   4, 30),
    ("MCS602", "Data Structures", 4, 25),
    ("MCS603", "Programming II",  4, 30),
]
cursor.executemany(
    "INSERT INTO courses (code, title, credits, capacity) VALUES (?,?,?,?)",
    courses
)

# Insert students
students = [
    ("Alice",   "alice@uni.edu",   "2003-05-12"),
    ("Bob",     "bob@uni.edu",     "2004-01-22"),
    ("Charlie", "charlie@uni.edu", "2003-11-30"),
    ("Diana",   "diana@uni.edu",   "2004-07-08"),
    ("Eve",     "eve@uni.edu",     "2003-09-15"),
]
cursor.executemany(
    "INSERT INTO students (name, email, dob) VALUES (?,?,?)",
    students
)

# Enroll students
enrollments = [
    (1, "MCS603", 92), (2, "MCS603", 75), (3, "MCS603", 88),
    (4, "MCS603", 65), (5, "MCS603", 95),
    (1, "MCS601", 90), (2, "MCS601", 80),
]
cursor.executemany(
    "INSERT INTO enrollments (student_id, course_code, grade) VALUES (?,?,?)",
    enrollments
)

conn.commit()
print("Data inserted.")

### READ — SELECT

In [ ]:
# All students
cursor.execute("SELECT id, name, email FROM students ORDER BY name")
rows = cursor.fetchall()

print(f"{'ID':>4}  {'Name':<12}  Email")
print("-" * 40)
for row in rows:
    print(f"{row['id']:>4}  {row['name']:<12}  {row['email']}")

In [ ]:
# Filter: students with grade >= 80 in MCS603
cursor.execute("""
    SELECT s.name, e.grade
    FROM   students s
    JOIN   enrollments e ON s.id = e.student_id
    WHERE  e.course_code = ? AND e.grade >= ?
    ORDER  BY e.grade DESC
""", ("MCS603", 80))

print("High performers in MCS603:")
for row in cursor.fetchall():
    print(f"  {row['name']:<12}  {row['grade']}")

In [ ]:
# Aggregate: stats per course
cursor.execute("""
    SELECT c.code, c.title,
           COUNT(e.id)    AS enrolled,
           AVG(e.grade)   AS avg_grade,
           MAX(e.grade)   AS highest,
           MIN(e.grade)   AS lowest
    FROM   courses c
    LEFT JOIN enrollments e ON c.code = e.course_code
    GROUP BY c.code
""")
print(f"{'Code':<8} {'Title':<20} {'Enrolled':>8} {'Avg':>6} {'High':>5} {'Low':>5}")
print("-" * 56)
for row in cursor.fetchall():
    avg = f"{row['avg_grade']:.1f}" if row['avg_grade'] else "N/A"
    print(f"{row['code']:<8} {row['title']:<20} {row['enrolled']:>8} {avg:>6} {row['highest'] or 'N/A':>5} {row['lowest'] or 'N/A':>5}")

### UPDATE

In [ ]:
# Update Bob's grade in MCS603
cursor.execute("""
    UPDATE enrollments
    SET    grade = ?
    WHERE  student_id = (SELECT id FROM students WHERE name = ?)
    AND    course_code = ?
""", (85, "Bob", "MCS603"))
conn.commit()

rows_changed = cursor.rowcount
print(f"{rows_changed} row(s) updated.")

# Verify
cursor.execute("SELECT s.name, e.grade FROM students s JOIN enrollments e ON s.id=e.student_id WHERE s.name='Bob' AND e.course_code='MCS603'")
print("Bob's new grade:", cursor.fetchone()["grade"])

### DELETE

In [ ]:
# Delete a specific enrollment
cursor.execute("""
    DELETE FROM enrollments
    WHERE student_id = (SELECT id FROM students WHERE name = ?)
    AND   course_code = ?
""", ("Bob", "MCS601"))
conn.commit()
print(f"Deleted {cursor.rowcount} enrollment(s).")

---
## 5. Parameterised Queries & SQL Injection

**NEVER** build SQL by string concatenation — it enables SQL injection attacks.

```python
# DANGEROUS — never do this
name = input()
cursor.execute("SELECT * FROM students WHERE name = '" + name + "'")
# If name = "'; DROP TABLE students; --"  → disaster!

# SAFE — always use parameterised queries
cursor.execute("SELECT * FROM students WHERE name = ?", (name,))
```

The `?` placeholder is escaped by sqlite3 — user input can never be interpreted as SQL.

In [ ]:
def find_student(name):
    cursor.execute(
        "SELECT id, name, email FROM students WHERE name LIKE ?",
        (f"%{name}%",)   # wrap in % for partial match
    )
    return cursor.fetchall()

results = find_student("ali")
for r in results:
    print(dict(r))

---
## 6. Transactions

A **transaction** groups multiple SQL operations — they either ALL succeed or ALL fail (atomicity).

Use `conn.commit()` to save and `conn.rollback()` to undo.

In [ ]:
def transfer_enrollment(student_name, from_course, to_course):
    try:
        # Get student ID
        cursor.execute("SELECT id FROM students WHERE name = ?", (student_name,))
        row = cursor.fetchone()
        if not row:
            raise ValueError(f"Student '{student_name}' not found")
        sid = row["id"]

        # Get existing grade
        cursor.execute(
            "SELECT grade FROM enrollments WHERE student_id=? AND course_code=?",
            (sid, from_course)
        )
        old = cursor.fetchone()
        if not old:
            raise ValueError(f"Not enrolled in {from_course}")

        cursor.execute(
            "DELETE FROM enrollments WHERE student_id=? AND course_code=?",
            (sid, from_course)
        )
        cursor.execute(
            "INSERT INTO enrollments (student_id, course_code, grade) VALUES (?,?,?)",
            (sid, to_course, old["grade"])
        )
        conn.commit()
        print(f"Transferred {student_name}: {from_course} → {to_course}")
    except Exception as e:
        conn.rollback()   # undo BOTH operations
        print(f"Transfer failed: {e}")

transfer_enrollment("Alice", "MCS601", "MCS602")
transfer_enrollment("Alice", "MCS999", "MCS602")  # should fail & rollback

---
## 7. Full Student Management System

A clean class-based wrapper around all CRUD operations.

In [ ]:
class StudentManager:
    def __init__(self, db_path="school.db"):
        self.conn = sqlite3.connect(db_path)
        self.conn.row_factory = sqlite3.Row
        self.cursor = self.conn.cursor()

    # ── Students ───────────────────────────────────────────────────
    def add_student(self, name, email, dob=None):
        try:
            self.cursor.execute(
                "INSERT INTO students (name, email, dob) VALUES (?,?,?)",
                (name, email, dob)
            )
            self.conn.commit()
            return self.cursor.lastrowid
        except sqlite3.IntegrityError as e:
            raise ValueError(f"Duplicate email or constraint error: {e}")

    def get_student(self, student_id):
        self.cursor.execute("SELECT * FROM students WHERE id=?", (student_id,))
        row = self.cursor.fetchone()
        return dict(row) if row else None

    def search_students(self, query):
        self.cursor.execute(
            "SELECT * FROM students WHERE name LIKE ? OR email LIKE ?",
            (f"%{query}%", f"%{query}%")
        )
        return [dict(r) for r in self.cursor.fetchall()]

    def update_student(self, student_id, **fields):
        allowed = {"name", "email", "dob"}
        updates = {k: v for k, v in fields.items() if k in allowed}
        if not updates:
            return 0
        set_clause = ", ".join(f"{k}=?" for k in updates)
        self.cursor.execute(
            f"UPDATE students SET {set_clause} WHERE id=?",
            (*updates.values(), student_id)
        )
        self.conn.commit()
        return self.cursor.rowcount

    def delete_student(self, student_id):
        self.cursor.execute("DELETE FROM students WHERE id=?", (student_id,))
        self.conn.commit()
        return self.cursor.rowcount

    # ── Grades ─────────────────────────────────────────────────────
    def set_grade(self, student_id, course_code, grade):
        self.cursor.execute("""
            INSERT INTO enrollments (student_id, course_code, grade)
            VALUES (?,?,?)
            ON CONFLICT(student_id, course_code) DO UPDATE SET grade=excluded.grade
        """, (student_id, course_code, grade))
        self.conn.commit()

    def class_stats(self, course_code):
        self.cursor.execute("""
            SELECT AVG(grade) avg, MAX(grade) high, MIN(grade) low, COUNT(*) total
            FROM   enrollments WHERE course_code=?
        """, (course_code,))
        return dict(self.cursor.fetchone())

    def close(self):
        self.conn.close()


# Demo
db = StudentManager("school.db")

new_id = db.add_student("Henry", "henry@uni.edu", "2004-03-20")
print("Added Henry with ID:", new_id)

db.set_grade(new_id, "MCS603", 88)

stats = db.class_stats("MCS603")
print(f"MCS603 stats: avg={stats['avg']:.1f}, high={stats['high']}, low={stats['low']}, total={stats['total']}")

db.close()

---
## 8. MySQL Connector (Reference)

The API is nearly identical to sqlite3.

```python
import mysql.connector

conn = mysql.connector.connect(
    host="localhost",
    user="root",
    password="secret",
    database="school"
)
cursor = conn.cursor(dictionary=True)   # returns dicts (like Row factory)

# Parameterised queries use %s instead of ?
cursor.execute("SELECT * FROM students WHERE name = %s", ("Alice",))

cursor.close()
conn.close()
```

### Key Differences: SQLite vs MySQL
| Feature | SQLite | MySQL |
|---|---|---|
| Setup | No server, single file | Needs server |
| Placeholder | `?` | `%s` |
| Auto PK | `AUTOINCREMENT` | `AUTO_INCREMENT` |
| Best for | Dev, embedded, testing | Production web apps |

---
## 9. Lab Exercises

### Lab 1: CRUD Practice
Using the `school.db` database created in this notebook:

1. Add 3 more students
2. Enrol them in at least 2 courses each with grades
3. Update one student's email
4. Delete one enrollment
5. Print a full report: each student, their courses, and grades

In [ ]:
import sqlite3
conn = sqlite3.connect("school.db")
conn.row_factory = sqlite3.Row
cursor = conn.cursor()

# TODO: all 5 tasks


### Lab 2: Advanced Queries
Write SQL queries to find:
1. Students enrolled in more than one course
2. The course with the highest average grade
3. Students who scored above the class average in MCS603
4. Students not enrolled in any course

In [ ]:
# TODO: 4 queries


---
## 10. Assignment — Week 12–13

**Database-Driven Student Management System**

Build a complete CLI application (`student_db_app.py`) using SQLite.

**Schema (20 pts)**  
Design tables: `students`, `courses`, `lecturers`, `enrollments`.  
Include appropriate PKs, FKs, NOT NULL constraints, and CHECK constraints.

**CRUD Operations (40 pts)**  
Implement for `students` and `enrollments`:
- Add / view / update / delete
- All operations must use parameterised queries
- Handle `IntegrityError` gracefully

**Reports (30 pts)**  
- Class report: all students, their courses, grades, letter grade
- Course statistics: avg, max, min, pass rate per course
- Dean's List: students with avg grade ≥ 80 across all courses

**Bulk Import (10 pts)**  
Read `students_import.csv` and insert all records in a single transaction.  
Rollback and log any rows that fail validation.

In [ ]:
# Plan schema and implement here


---
## Summary — Week 12–13

| Concept | Key Point |
|---|---|
| `sqlite3.connect` | Creates/opens `.db` file; `:memory:` for testing |
| `cursor.execute` | Run one SQL statement |
| `cursor.executemany` | Run same statement for a list of rows |
| Parameterised queries | Use `?` placeholders — prevents SQL injection |
| `conn.commit` | Persist changes |
| `conn.rollback` | Undo uncommitted changes |
| `row_factory` | Set to `sqlite3.Row` for dict-style access |
| MySQL | Use `mysql-connector-python`; `%s` placeholders |

**Next:** Week 14 — Capstone Project